# Scientific Knowledge Graph Comparison

Курсовой проект: сравнительный анализ методов построения графов научных знаний с использованием эмбеддингов и локальных LLM.

In [ ]:
# 1. Установка зависимостей
# В Kaggle можно выполнить эту ячейку, если пакеты отсутствуют.
# !pip install -q pandas numpy PyMuPDF sentence-transformers scikit-learn networkx pyvis matplotlib transformers torch accelerate tqdm

In [ ]:
# No Hugging Face API token is required.
# The LLM is loaded locally with transformers in the Colab runtime.

In [ ]:
import os

# 2. Конфигурация
import os
from pathlib import Path
import sys

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
HF_ONLINE = True
if HF_ONLINE:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)
else:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

PAPERS_PATH = PROJECT_ROOT / "data" / "papers"
RESULTS_PATH = PROJECT_ROOT / "data" / "results"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
SIMILARITY_THRESHOLD = 0.55
TOP_K = 5
LLM_SIMILARITY_THRESHOLD = 0.50
LLM_TOP_K = 10
USE_LLM = True
LLM_MAX_PAIRS = None

Path(RESULTS_PATH).mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Papers path:", PAPERS_PATH)

In [ ]:
# 3. Импорты проекта
import pandas as pd
from tqdm.auto import tqdm

from src.pdf_parser import parse_papers_from_folder
from src.preprocessing import clean_text, truncate_text
from src.embeddings import load_embedding_model, compute_embeddings, compute_similarity_matrix, get_candidate_pairs
from src.citation_extractor import extract_references, match_references_to_papers
from src.graph_builder import build_similarity_graph, build_citation_graph, build_llm_graph, build_hybrid_graph
from src.graph_metrics import calculate_graph_metrics, compare_graphs
from src.visualization import visualize_graph_pyvis, plot_metric_comparison
from src.utils import save_dataframe, save_json, save_graph_gexf

In [ ]:
# 4. Загрузка и парсинг PDF
papers_df = parse_papers_from_folder(PAPERS_PATH)
papers_df["text_clean"] = papers_df["text"].fillna("").apply(clean_text)
papers_df["text_for_embedding"] = papers_df.apply(
    lambda row: truncate_text(row["abstract"] if isinstance(row["abstract"], str) and row["abstract"].strip() else row["text_clean"], 5000),
    axis=1,
)
save_dataframe(papers_df, Path(RESULTS_PATH) / "papers_processed.csv")
papers_df[["paper_id", "filename", "title", "abstract"]].head(20)

In [ ]:
# 5. Embeddings и similarity matrix
embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)
embeddings = compute_embeddings(papers_df["text_for_embedding"].tolist(), embedding_model)
similarity_matrix = compute_similarity_matrix(embeddings)
candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=SIMILARITY_THRESHOLD, top_k=TOP_K)
llm_candidate_pairs = get_candidate_pairs(similarity_matrix, threshold=LLM_SIMILARITY_THRESHOLD, top_k=LLM_TOP_K)
candidate_pairs[:5], len(candidate_pairs), len(llm_candidate_pairs)

In [ ]:
# 6. Similarity Graph
similarity_graph = build_similarity_graph(papers_df, candidate_pairs)
save_graph_gexf(similarity_graph, Path(RESULTS_PATH) / "similarity_graph.gexf")
print(similarity_graph)

In [ ]:
# 7. Citation edges и Citation Graph
citation_edges = []
seen_edges = set()

for _, row in tqdm(papers_df.iterrows(), total=len(papers_df), desc="Matching citations"):
    references_text = row.get("references_raw", "") or row.get("text_clean", "")
    references = extract_references(references_text)
    corpus_with_source = papers_df.copy()
    corpus_with_source.attrs["source_paper_id"] = int(row["paper_id"])
    for edge in match_references_to_papers(references, corpus_with_source):
        key = (edge["source"], edge["target"])
        if edge["source"] != -1 and key not in seen_edges:
            citation_edges.append(edge)
            seen_edges.add(key)

citation_graph = build_citation_graph(papers_df, citation_edges)
save_graph_gexf(citation_graph, Path(RESULTS_PATH) / "citation_graph.gexf")
pd.DataFrame(citation_edges).head(), len(citation_edges)

In [ ]:
# 8. LLM Relation Graph, опционально
llm_edges = []
llm_results = []

if USE_LLM and candidate_pairs:
    import importlib
    import src.llm_relation_extractor as llm_relation_extractor
    importlib.reload(llm_relation_extractor)
    load_llm = llm_relation_extractor.load_llm
    classify_relation = llm_relation_extractor.classify_relation
    llm, tokenizer = load_llm(LLM_MODEL_NAME)
    papers_by_id = papers_df.set_index("paper_id").to_dict(orient="index")
    pairs_for_llm = llm_candidate_pairs[:LLM_MAX_PAIRS] if LLM_MAX_PAIRS else llm_candidate_pairs
    for pair in tqdm(pairs_for_llm, desc="LLM relation extraction"):
        paper_a = {"paper_id": pair["source"], **papers_by_id[pair["source"]]}
        paper_b = {"paper_id": pair["target"], **papers_by_id[pair["target"]]}
        result = classify_relation(paper_a, paper_b, llm, tokenizer)
        result["candidate_similarity"] = pair.get("similarity")
        llm_results.append(result)
        if result["relation"] != "NO_RELATION":
            llm_edges.append(result)
else:
    print("USE_LLM=False: LLM graph will be empty but the rest of the pipeline works.")

llm_results_df = pd.DataFrame(llm_results)
llm_edges_df = pd.DataFrame(llm_edges)
if not llm_results_df.empty:
    print(llm_results_df["relation"].value_counts())

llm_graph = build_llm_graph(papers_df, llm_edges)
save_dataframe(llm_results_df, Path(RESULTS_PATH) / "llm_relation_results.csv")
save_graph_gexf(llm_graph, Path(RESULTS_PATH) / "llm_graph.gexf")
llm_edges_df.head(), len(llm_edges), len(llm_results)

In [ ]:
# 9. Hybrid Graph
hybrid_graph = build_hybrid_graph(papers_df, candidate_pairs, citation_edges, llm_edges)
save_graph_gexf(hybrid_graph, Path(RESULTS_PATH) / "hybrid_graph.gexf")
print(hybrid_graph)

In [ ]:
# 10. Метрики и сравнительная таблица
graphs = {
    "Embedding Similarity Graph": similarity_graph,
    "Citation Graph": citation_graph,
    "LLM Relation Graph": llm_graph,
    "Hybrid Graph": hybrid_graph,
}

metrics = {name: calculate_graph_metrics(graph) for name, graph in graphs.items()}
metrics_df = compare_graphs(graphs)
save_dataframe(metrics_df, Path(RESULTS_PATH) / "graph_metrics_comparison.csv")
save_json(metrics, Path(RESULTS_PATH) / "graph_metrics_full.json")
metrics_df

In [ ]:
# 11. Визуализация графов
for name, graph in graphs.items():
    safe_name = name.lower().replace(" ", "_")
    visualize_graph_pyvis(graph, str(Path(RESULTS_PATH) / f"{safe_name}.html"))

for metric_name in ["number_of_edges", "density", "average_degree", "average_clustering", "number_of_communities"]:
    plot_metric_comparison(metrics_df, metric_name, str(Path(RESULTS_PATH) / f"compare_{metric_name}.png"))

print("Saved visualizations to", RESULTS_PATH)

In [ ]:
import pandas as pd
df=pd.read_csv("data/results/llm_relation_results.csv")

## 12. Краткие выводы

В итоговой таблице сравниваются четыре способа построения ребер. Обычно similarity graph дает более плотную и связную структуру, citation graph отражает явные библиографические зависимости и может быть разреженным, LLM graph добавляет типизированные семантические отношения только для candidate pairs, а hybrid graph объединяет преимущества всех источников связей. Главный анализ курсовой работы: объяснить, как выбор метода построения ребер меняет плотность, связность, центральные статьи, PageRank и community structure.